# Tahap 9 — Label Audit & Modeling Readiness Assessment

Notebook ini bersifat **audit dan evaluasi murni**. Tidak ada training model, feature engineering baru,
lookback generation, balancing, oversampling, undersampling, scaling, normalisasi, maupun penghapusan
record yang dilakukan pada notebook ini.

**Input:** `08_labeled_dataset.csv`
**Output:**
- `09_modeling_readiness.csv`
- `STAGE9_REPORT.md`

Notebook bersifat reproducible: menjalankan seluruh cell dari awal menggunakan file input akan
menghasilkan output yang identik.


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load Dataset

In [2]:
INPUT_PATH = '/mnt/user-data/outputs/08_labeled_dataset.csv'
OUTPUT_DIR = '/mnt/user-data/outputs'
OUTPUT_CSV_PATH = os.path.join(OUTPUT_DIR, '09_modeling_readiness.csv')
OUTPUT_REPORT_PATH = os.path.join(OUTPUT_DIR, 'STAGE9_REPORT.md')

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_original = pd.read_csv(INPUT_PATH)
df = df_original.copy()  # copy untuk audit; df_original tetap tidak tersentuh sebagai bukti "tidak ada data yang diubah"

n_rows_input = len(df_original)
n_cols_input = df_original.shape[1]

print("Shape:", df.shape)
df.head()


Shape: (2922, 15)


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape,missing_group,risk_label,label_source
0,2017-01-01,12Z,SELECTED,26.9,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401,IMPUTED_OGIMET,Sedang,RR_ONLY
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164,ORIGINAL,Sedang,RR_ONLY
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572,ORIGINAL,Tinggi,RR_ONLY
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409,ORIGINAL,Sedang,RR_ONLY
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721,ORIGINAL,Tinggi,RR_ONLY


## 3. Dataset Summary (ANALYSIS 1)

Melaporkan jumlah row, kolom, rentang tanggal, jumlah feature, jumlah kelas label, dan missing value per kolom.

In [3]:
n_rows, n_cols = df.shape
date_min, date_max = df['date'].min(), df['date'].max()
n_classes = df['risk_label'].nunique()

print("Jumlah row  :", n_rows)
print("Jumlah kolom:", n_cols)
print("Rentang tanggal:", date_min, "s.d.", date_max)
print("Jumlah kelas risk_label:", n_classes)
print("Kelas:", sorted(df['risk_label'].unique()))


Jumlah row  : 2922
Jumlah kolom: 15
Rentang tanggal: 2017-01-01 s.d. 2024-12-31
Jumlah kelas risk_label: 4
Kelas: ['Rendah', 'Sangat Tinggi', 'Sedang', 'Tinggi']


In [4]:
# Fitur model sesuai BAB III Subbab 3.7.4 (CAPE dikecualikan, hanya untuk pembentukan label)
expected_features_bab3 = ['rr', 'tavg', 'rh', 'cin', 'kindex', 'li', 'tt', 'sweat', 'pw']
present_features = [c for c in expected_features_bab3 if c in df.columns]
missing_features = [c for c in expected_features_bab3 if c not in df.columns]

print("Fitur yang tersedia di dataset  :", present_features, f"({len(present_features)} fitur)")
print("Fitur yang disebut BAB III tapi TIDAK ada di dataset:", missing_features)


Fitur yang tersedia di dataset  : ['rr', 'tavg', 'rh', 'cin', 'kindex', 'li', 'tt', 'sweat'] (8 fitur)
Fitur yang disebut BAB III tapi TIDAK ada di dataset: ['pw']


In [5]:
missing_per_col = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_per_col,
    'missing_pct': missing_pct
})
missing_summary


,missing_count,missing_pct
date,0,0.00
selected_hour,0,0.00
selection_status,0,0.00
rr,0,0.00
tavg,0,0.00
rh,0,0.00
cin,148,5.07
kindex,148,5.07
li,148,5.07
tt,148,5.07


## 4. Label Distribution Audit (ANALYSIS 2)

Menghitung jumlah, persentase, dan class imbalance ratio pada `risk_label`, serta menentukan kelas mayoritas
dan minoritas.

In [6]:
label_counts = df['risk_label'].value_counts()
label_pct = (df['risk_label'].value_counts(normalize=True) * 100).round(2)

label_distribution = pd.DataFrame({
    'jumlah': label_counts,
    'persentase': label_pct
})

majority_class = label_counts.idxmax()
minority_class = label_counts.idxmin()
imbalance_ratio = label_counts.max() / label_counts.min()

print("Kelas mayoritas :", majority_class, f"({label_counts.max()} record)")
print("Kelas minoritas :", minority_class, f"({label_counts.min()} record)")
print("Imbalance ratio (mayoritas:minoritas):", round(imbalance_ratio, 2))
print()
label_distribution


Kelas mayoritas : Rendah (2331 record)
Kelas minoritas : Sangat Tinggi (45 record)
Imbalance ratio (mayoritas:minoritas): 51.8



,jumlah,persentase
risk_label,,
Rendah,2331,79.77
Sedang,372,12.73
Tinggi,174,5.95
Sangat Tinggi,45,1.54


In [7]:
print("Rasio setiap kelas terhadap kelas mayoritas:")
class_order = ["Rendah", "Sedang", "Tinggi", "Sangat Tinggi"]
imbalance_per_class = {}
for cls in class_order:
    if cls in label_counts.index:
        ratio = label_counts.max() / label_counts[cls]
        imbalance_per_class[cls] = round(ratio, 2)
        print(f"  {cls}: 1 : {ratio:.2f}")


Rasio setiap kelas terhadap kelas mayoritas:
  Rendah: 1 : 1.00
  Sedang: 1 : 6.27
  Tinggi: 1 : 13.40
  Sangat Tinggi: 1 : 51.80


## 5. Label Source Audit (ANALYSIS 3)

Menghitung jumlah dan persentase `label_source`, serta kontribusi CAPE terhadap seluruh dataset.

In [8]:
source_counts = df['label_source'].value_counts()
source_pct = (df['label_source'].value_counts(normalize=True) * 100).round(2)

label_source_summary = pd.DataFrame({
    'jumlah': source_counts,
    'persentase': source_pct
})
label_source_summary


,jumlah,persentase
label_source,,
RR_ONLY,2881,98.60
RR_CAPE_MODIFIED,24,0.82
RR_ONLY_CAPE_MISSING,17,0.58


In [9]:
cape_modified_count = int(source_counts.get('RR_CAPE_MODIFIED', 0))
cape_missing_count = int(source_counts.get('RR_ONLY_CAPE_MISSING', 0))
rr_only_count = int(source_counts.get('RR_ONLY', 0))

cape_contribution_pct = cape_modified_count / n_rows * 100

print(f"RR_ONLY               : {rr_only_count} ({rr_only_count/n_rows*100:.2f}%)")
print(f"RR_CAPE_MODIFIED       : {cape_modified_count} ({cape_modified_count/n_rows*100:.2f}%)")
print(f"RR_ONLY_CAPE_MISSING   : {cape_missing_count} ({cape_missing_count/n_rows*100:.2f}%)")
print()
print(f"Kontribusi CAPE terhadap seluruh dataset (mengubah label): {cape_contribution_pct:.2f}%")


RR_ONLY               : 2881 (98.60%)
RR_CAPE_MODIFIED       : 24 (0.82%)
RR_ONLY_CAPE_MISSING   : 17 (0.58%)

Kontribusi CAPE terhadap seluruh dataset (mengubah label): 0.82%


## 6. CAPE Impact Assessment (ANALYSIS 4)

Mengevaluasi jumlah label yang berubah karena CAPE, distribusi perubahan per kelas, dan signifikansi dampaknya.

In [10]:
def base_label(rr):
    if rr < 20:
        return "Rendah"
    elif rr < 50:
        return "Sedang"
    elif rr < 100:
        return "Tinggi"
    else:
        return "Sangat Tinggi"

def in_boundary(rr):
    return (15 <= rr <= 25) or (45 <= rr <= 55) or (95 <= rr <= 105)

modified = df[df['label_source'] == 'RR_CAPE_MODIFIED'].copy()
n_modified = len(modified)

boundary_total = int(df['rr'].apply(in_boundary).sum())

pct_of_total = n_modified / n_rows * 100
pct_of_boundary = n_modified / boundary_total * 100 if boundary_total > 0 else np.nan

print("Jumlah label berubah karena CAPE:", n_modified)
print(f"  - sebagai % dari total dataset          : {pct_of_total:.2f}%")
print(f"  - sebagai % dari record boundary zone    : {pct_of_boundary:.2f}% (dari {boundary_total} record boundary)")


Jumlah label berubah karena CAPE: 24
  - sebagai % dari total dataset          : 0.82%
  - sebagai % dari record boundary zone    : 8.30% (dari 289 record boundary)


In [11]:
modified_base = modified['rr'].apply(base_label)
direction = modified_base + " -> " + modified['risk_label']
direction_counts = direction.value_counts()

print("Distribusi arah perubahan label akibat CAPE:")
print(direction_counts)


Distribusi arah perubahan label akibat CAPE:
Sedang -> Tinggi           13
Rendah -> Sedang           10
Tinggi -> Sangat Tinggi     1
Name: count, dtype: int64


In [12]:
print("Kontribusi CAPE terhadap komposisi tiap kelas akhir:")
cape_class_impact = []
for cls in class_order:
    total_cls = int((df['risk_label'] == cls).sum())
    from_cape = int((modified['risk_label'] == cls).sum())
    pct = (from_cape / total_cls * 100) if total_cls > 0 else 0
    cape_class_impact.append({
        'risk_label': cls,
        'total_record': total_cls,
        'dari_modifikasi_cape': from_cape,
        'persentase_dari_kelas': round(pct, 2)
    })
    print(f"  {cls:15s}: total={total_cls:5d}  dari_CAPE={from_cape:3d}  ({pct:.2f}% dari kelas)")

cape_class_impact_df = pd.DataFrame(cape_class_impact)
cape_class_impact_df


Kontribusi CAPE terhadap komposisi tiap kelas akhir:
  Rendah         : total= 2331  dari_CAPE=  0  (0.00% dari kelas)
  Sedang         : total=  372  dari_CAPE= 10  (2.69% dari kelas)
  Tinggi         : total=  174  dari_CAPE= 13  (7.47% dari kelas)
  Sangat Tinggi  : total=   45  dari_CAPE=  1  (2.22% dari kelas)


,risk_label,total_record,dari_modifikasi_cape,persentase_dari_kelas
0,Rendah,2331,0,0.00
1,Sedang,372,10,2.69
2,Tinggi,174,13,7.47
3,Sangat Tinggi,45,1,2.22


In [13]:
# Interpretasi signifikansi dampak CAPE
print("=== INTERPRETASI ===")
print(f"- CAPE mengubah label pada {pct_of_total:.2f}% dari total dataset -> dampak AGREGAT sangat kecil (<1%).")
print(f"- Namun secara relatif per kelas, CAPE berkontribusi hingga "
      f"{cape_class_impact_df['persentase_dari_kelas'].max():.2f}% pada kelas "
      f"'{cape_class_impact_df.loc[cape_class_impact_df['persentase_dari_kelas'].idxmax(), 'risk_label']}'.")
print("- Kesimpulan: CAPE TIDAK signifikan secara statistik terhadap distribusi label keseluruhan,")
print("  namun berperan konsisten sebagai conditional risk modifier sesuai desain metodologi BAB III")
print("  (bukan indikator utama, tidak mendominasi/mendistorsi distribusi kelas).")


=== INTERPRETASI ===
- CAPE mengubah label pada 0.82% dari total dataset -> dampak AGREGAT sangat kecil (<1%).
- Namun secara relatif per kelas, CAPE berkontribusi hingga 7.47% pada kelas 'Tinggi'.
- Kesimpulan: CAPE TIDAK signifikan secara statistik terhadap distribusi label keseluruhan,
  namun berperan konsisten sebagai conditional risk modifier sesuai desain metodologi BAB III
  (bukan indikator utama, tidak mendominasi/mendistorsi distribusi kelas).


## 7. Modeling Readiness Assessment (ANALYSIS 5)

Evaluasi kesiapan dataset untuk LSTM multiclass, risiko class imbalance, estimasi dampak lookback terhadap
jumlah sampel, dan potensi hilangnya sampel minoritas. **Tidak ada sequence yang dibuat** — hanya analisis
kuantitatif/estimasi.

In [14]:
df['date_dt'] = pd.to_datetime(df['date'])

full_range = pd.date_range(df['date_dt'].min(), df['date_dt'].max(), freq='D')
missing_dates = full_range.difference(df['date_dt'])

print("Total tanggal kalender yang diharapkan (harian, tanpa gap):", len(full_range))
print("Total tanggal unik pada dataset                            :", df['date_dt'].nunique())
print("Jumlah tanggal kalender yang hilang (gap)                  :", len(missing_dates))


Total tanggal kalender yang diharapkan (harian, tanpa gap): 2922
Total tanggal unik pada dataset                            : 2922
Jumlah tanggal kalender yang hilang (gap)                  : 0


In [15]:
df['year'] = df['date_dt'].dt.year

print("Distribusi risk_label per tahun:")
year_dist = pd.crosstab(df['year'], df['risk_label'])
year_dist


Distribusi risk_label per tahun:


risk_label,Rendah,Sangat Tinggi,Sedang,Tinggi
year,,,,
2017,287,2,48,28
2018,294,1,47,23
2019,314,0,38,13
2020,296,7,43,20
2021,286,9,45,25
2022,282,13,52,18
2023,301,5,41,18
2024,271,8,58,29


In [16]:
print("Distribusi kejadian 'Sangat Tinggi' per tahun:")
st_per_year = df.loc[df['risk_label'] == 'Sangat Tinggi', 'year'].value_counts().sort_index()
print(st_per_year)

st_dates = df.loc[df['risk_label'] == 'Sangat Tinggi', 'date_dt'].sort_values()
gaps = st_dates.diff().dt.days.dropna()

print()
print("Jarak antar kejadian 'Sangat Tinggi' (hari) - statistik deskriptif:")
print(gaps.describe())


Distribusi kejadian 'Sangat Tinggi' per tahun:
year
2017     2
2018     1
2020     7
2021     9
2022    13
2023     5
2024     8
Name: count, dtype: int64

Jarak antar kejadian 'Sangat Tinggi' (hari) - statistik deskriptif:
count     44.000000
mean      63.022727
std      110.961022
min        1.000000
25%       14.000000
50%       34.500000
75%       61.750000
max      713.000000
Name: date_dt, dtype: float64


In [17]:
print("=== ESTIMASI DAMPAK LOOKBACK WINDOW TERHADAP JUMLAH SAMPEL ===")
lookback_impact = []
for W in [7, 14, 30, 60]:
    usable = n_rows - W
    lost = W
    lost_pct = lost / n_rows * 100
    lookback_impact.append({'lookback_window': W, 'usable_sequences': usable, 'sampel_hilang': lost, 'persen_hilang': round(lost_pct, 2)})
    print(f"Lookback window={W:2d} hari: usable sequences={usable}  (hilang {lost}, {lost_pct:.2f}% dari total)")

lookback_impact_df = pd.DataFrame(lookback_impact)


=== ESTIMASI DAMPAK LOOKBACK WINDOW TERHADAP JUMLAH SAMPEL ===
Lookback window= 7 hari: usable sequences=2915  (hilang 7, 0.24% dari total)
Lookback window=14 hari: usable sequences=2908  (hilang 14, 0.48% dari total)
Lookback window=30 hari: usable sequences=2892  (hilang 30, 1.03% dari total)
Lookback window=60 hari: usable sequences=2862  (hilang 60, 2.05% dari total)


In [18]:
print("=== ESTIMASI DAMPAK LOOKBACK PADA SKEMA 3-WAY CHRONOLOGICAL SPLIT ===")
lookback_split_impact = []
for W in [7, 14, 30]:
    total_loss = 3 * W  # loss di awal train, awal val, awal test
    total_loss_pct = total_loss / n_rows * 100
    lookback_split_impact.append({'lookback_window': W, 'estimasi_sampel_hilang_3_split': total_loss, 'persen_hilang': round(total_loss_pct, 2)})
    print(f"Lookback window={W:2d} hari: estimasi total sampel hilang (3 titik potong)={total_loss} ({total_loss_pct:.2f}%)")

lookback_split_impact_df = pd.DataFrame(lookback_split_impact)


=== ESTIMASI DAMPAK LOOKBACK PADA SKEMA 3-WAY CHRONOLOGICAL SPLIT ===
Lookback window= 7 hari: estimasi total sampel hilang (3 titik potong)=21 (0.72%)
Lookback window=14 hari: estimasi total sampel hilang (3 titik potong)=42 (1.44%)
Lookback window=30 hari: estimasi total sampel hilang (3 titik potong)=90 (3.08%)


In [19]:
print("=== SIMULASI PEMBAGIAN KRONOLOGIS 70/15/15 (ILUSTRATIF, TIDAK DITERAPKAN) ===")
df_sorted = df.sort_values('date_dt').reset_index(drop=True)
N = len(df_sorted)
train_end = int(N * 0.70)
val_end = int(N * 0.85)

train_sim = df_sorted.iloc[:train_end]
val_sim = df_sorted.iloc[train_end:val_end]
test_sim = df_sorted.iloc[val_end:]

split_summary = []
for name, subset in [('Train', train_sim), ('Validation', val_sim), ('Test', test_sim)]:
    counts = subset['risk_label'].value_counts()
    row = {
        'subset': name,
        'jumlah_baris': len(subset),
        'tanggal_mulai': subset['date_dt'].min().date().isoformat(),
        'tanggal_akhir': subset['date_dt'].max().date().isoformat(),
    }
    for cls in class_order:
        row[cls] = int(counts.get(cls, 0))
    split_summary.append(row)
    print(f"{name:10s}: {len(subset):4d} baris ({subset['date_dt'].min().date()} - {subset['date_dt'].max().date()})")
    print("   ", dict(counts))

split_summary_df = pd.DataFrame(split_summary)
split_summary_df


=== SIMULASI PEMBAGIAN KRONOLOGIS 70/15/15 (ILUSTRATIF, TIDAK DITERAPKAN) ===
Train     : 2045 baris (2017-01-01 - 2022-08-07)
    {'Rendah': np.int64(1659), 'Sedang': np.int64(246), 'Tinggi': np.int64(115), 'Sangat Tinggi': np.int64(25)}
Validation:  438 baris (2022-08-08 - 2023-10-19)
    {'Rendah': np.int64(335), 'Sedang': np.int64(63), 'Tinggi': np.int64(28), 'Sangat Tinggi': np.int64(12)}
Test      :  439 baris (2023-10-20 - 2024-12-31)
    {'Rendah': np.int64(337), 'Sedang': np.int64(63), 'Tinggi': np.int64(31), 'Sangat Tinggi': np.int64(8)}


,subset,jumlah_baris,tanggal_mulai,tanggal_akhir,Rendah,Sedang,Tinggi,Sangat Tinggi
0,Train,2045,2017-01-01,2022-08-07,1659,246,115,25
1,Validation,438,2022-08-08,2023-10-19,335,63,28,12
2,Test,439,2023-10-20,2024-12-31,337,63,31,8


## 8. Training Recommendation (ANALYSIS 6)

Rekomendasi bersifat analitis/rencana. **Tidak diterapkan** pada notebook ini.

In [20]:
recommendations = {
    "class_weight": "Direkomendasikan. Imbalance ratio 51,8:1 (Rendah vs Sangat Tinggi) membuat "
        "penggunaan class_weight (mis. compute_class_weight('balanced')) pada loss "
        "categorical crossentropy menjadi langkah minimal yang disarankan, tanpa mengubah data.",
    "focal_loss": "Dipertimbangkan sebagai opsi lanjutan jika class_weight belum cukup meningkatkan "
        "recall kelas Tinggi/Sangat Tinggi pada evaluasi model nanti.",
    "oversampling": "Tidak direkomendasikan diterapkan langsung pada data time-series mentah karena "
        "berisiko merusak struktur temporal/autokorelasi. Jika diperlukan, sebaiknya diterapkan "
        "pada level sequence (pasca look-back window) dengan teknik yang menjaga urutan waktu.",
    "split_strategy": "Split kronologis (bukan random) wajib digunakan sesuai BAB III 3.7.6. Simulasi "
        "70/15/15 menunjukkan kelas minoritas masih terwakili di semua subset, namun sangat tipis "
        "pada validation/test (8-12 sampel Sangat Tinggi) sehingga metrik evaluasi kelas ini berisiko "
        "bervarians tinggi. Rencana awal menggunakan data 2025 sebagai test set (BAB III) belum dapat "
        "dipenuhi karena dataset hanya sampai 2024-12-31.",
}

for k, v in recommendations.items():
    print(f"[{k}]")
    print(" ", v)
    print()


[class_weight]
  Direkomendasikan. Imbalance ratio 51,8:1 (Rendah vs Sangat Tinggi) membuat penggunaan class_weight (mis. compute_class_weight('balanced')) pada loss categorical crossentropy menjadi langkah minimal yang disarankan, tanpa mengubah data.

[focal_loss]
  Dipertimbangkan sebagai opsi lanjutan jika class_weight belum cukup meningkatkan recall kelas Tinggi/Sangat Tinggi pada evaluasi model nanti.

[oversampling]
  Tidak direkomendasikan diterapkan langsung pada data time-series mentah karena berisiko merusak struktur temporal/autokorelasi. Jika diperlukan, sebaiknya diterapkan pada level sequence (pasca look-back window) dengan teknik yang menjaga urutan waktu.

[split_strategy]
  Split kronologis (bukan random) wajib digunakan sesuai BAB III 3.7.6. Simulasi 70/15/15 menunjukkan kelas minoritas masih terwakili di semua subset, namun sangat tipis pada validation/test (8-12 sampel Sangat Tinggi) sehingga metrik evaluasi kelas ini berisiko bervarians tinggi. Rencana awal menggu

## 9. Export Output (`09_modeling_readiness.csv`)

In [21]:
majority_count = label_counts.max()

readiness_rows = []
for cls in class_order:
    sub = df[df['risk_label'] == cls]
    n = len(sub)
    pct = n / n_rows * 100
    imb_ratio = majority_count / n if n > 0 else np.nan
    src_counts = sub['label_source'].value_counts()
    rr_only_n = int(src_counts.get('RR_ONLY', 0))
    rr_cape_mod_n = int(src_counts.get('RR_CAPE_MODIFIED', 0))
    rr_cape_missing_n = int(src_counts.get('RR_ONLY_CAPE_MISSING', 0))
    pct_from_cape = (rr_cape_mod_n / n * 100) if n > 0 else 0
    readiness_rows.append({
        'risk_label': cls,
        'jumlah': n,
        'persentase': round(pct, 2),
        'imbalance_ratio_terhadap_mayoritas': round(imb_ratio, 2),
        'rr_only': rr_only_n,
        'rr_cape_modified': rr_cape_mod_n,
        'rr_only_cape_missing': rr_cape_missing_n,
        'persentase_dari_cape_modified': round(pct_from_cape, 2),
    })

readiness_df = pd.DataFrame(readiness_rows)

total_row = {
    'risk_label': 'TOTAL',
    'jumlah': n_rows,
    'persentase': 100.0,
    'imbalance_ratio_terhadap_mayoritas': np.nan,
    'rr_only': int((df['label_source'] == 'RR_ONLY').sum()),
    'rr_cape_modified': int((df['label_source'] == 'RR_CAPE_MODIFIED').sum()),
    'rr_only_cape_missing': int((df['label_source'] == 'RR_ONLY_CAPE_MISSING').sum()),
    'persentase_dari_cape_modified': round((df['label_source'] == 'RR_CAPE_MODIFIED').sum() / n_rows * 100, 2),
}
readiness_df = pd.concat([readiness_df, pd.DataFrame([total_row])], ignore_index=True)

readiness_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Disimpan ke: {OUTPUT_CSV_PATH}")
readiness_df


Disimpan ke: /mnt/user-data/outputs/09_modeling_readiness.csv


,risk_label,jumlah,persentase,imbalance_ratio_terhadap_mayoritas,rr_only,rr_cape_modified,rr_only_cape_missing,persentase_dari_cape_modified
0,Rendah,2331,79.77,1.00,2323,0,8,0.00
1,Sedang,372,12.73,6.27,355,10,7,2.69
2,Tinggi,174,5.95,13.40,159,13,2,7.47
3,Sangat Tinggi,45,1.54,51.80,44,1,0,2.22
4,TOTAL,2922,100.00,NaN,2881,24,17,0.82


## 10. Generate Report (`STAGE9_REPORT.md`)

In [22]:
def fmt_int(x):
    return f"{x:,}".replace(",", ".")

missing_features_line = ", ".join(f"`{c}`" for c in missing_features) if missing_features else "(tidak ada)"
present_features_line = ", ".join(f"`{c}`" for c in present_features)

missing_rows_md = "\n".join(
    f"| {col} | {int(missing_summary.loc[col, 'missing_count'])} | {missing_summary.loc[col, 'missing_pct']:.2f}% |"
    for col in df_original.columns if col in missing_summary.index
)

label_dist_rows = "\n".join(
    f"| {cls} | {fmt_int(int(label_counts[cls]))} | {label_pct[cls]:.2f}% |"
    for cls in class_order if cls in label_counts.index
)

imbalance_rows = "\n".join(
    f"| {cls} | 1 : {imbalance_per_class[cls]:.2f} |"
    for cls in class_order if cls in imbalance_per_class
)

source_rows = "\n".join(
    f"| {src} | {fmt_int(int(source_counts[src]))} | {source_pct[src]:.2f}% |"
    for src in ["RR_ONLY", "RR_CAPE_MODIFIED", "RR_ONLY_CAPE_MISSING"] if src in source_counts.index
)

direction_order = ["Rendah -> Sedang", "Sedang -> Tinggi", "Tinggi -> Sangat Tinggi"]
direction_rows = "\n".join(
    f"| {d} | {int(direction_counts.get(d, 0))} |"
    for d in direction_order if d in direction_counts.index
)

cape_class_rows = "\n".join(
    f"| {r['risk_label']} | {fmt_int(int(r['total_record']))} | {int(r['dari_modifikasi_cape'])} | {r['persentase_dari_kelas']:.2f}% |"
    for _, r in cape_class_impact_df.iterrows() if r['risk_label'] in ["Sedang", "Tinggi", "Sangat Tinggi"]
)

lookback_rows = "\n".join(
    f"| {r['lookback_window']} hari | {fmt_int(int(r['usable_sequences']))} | {int(r['sampel_hilang'])} | {r['persen_hilang']:.2f}% |"
    for _, r in lookback_impact_df.iterrows()
)

lookback_split_rows = "\n".join(
    f"| {r['lookback_window']} hari | {int(r['estimasi_sampel_hilang_3_split'])} | {r['persen_hilang']:.2f}% |"
    for _, r in lookback_split_impact_df.iterrows()
)

split_table_rows = "\n".join(
    f"| {r['subset']} | {fmt_int(int(r['jumlah_baris']))} | {r['tanggal_mulai']} s.d. {r['tanggal_akhir']} | "
    f"{fmt_int(int(r['Rendah']))} | {fmt_int(int(r['Sedang']))} | {fmt_int(int(r['Tinggi']))} | {fmt_int(int(r['Sangat Tinggi']))} |"
    for _, r in split_summary_df.iterrows()
)

max_impact_row = cape_class_impact_df.loc[cape_class_impact_df['persentase_dari_kelas'].idxmax()]

report = f"""# STAGE 9 REPORT - Label Audit & Modeling Readiness Assessment

**Input:** `08_labeled_dataset.csv` ({fmt_int(n_rows_input)} baris, {n_cols_input} kolom)
**Sifat tahap:** Audit dan evaluasi - **tidak ada data, label, atau baris yang diubah/dihapus.**
**Sumber metodologi:** BAB III - Subbab 3.7.3 (Pembentukan Label), 3.7.4 (Rekayasa Fitur), 3.7.6 (Pembagian Dataset), 3.7.8 (Pelatihan Model LSTM)

---

## ANALYSIS 1 - Dataset Summary

| Item | Nilai |
|---|---|
| Jumlah row | {fmt_int(n_rows)} |
| Jumlah kolom | {n_cols} |
| Rentang tanggal | {date_min} s.d. {date_max} (harian, tanpa gap kalender) |
| Jumlah feature model (kandidat, sesuai BAB III 3.7.4) | {len(present_features)} dari {len(expected_features_bab3)} yang disebutkan dalam BAB III ({present_features_line}) |
| Jumlah kelas label (`risk_label`) | {n_classes} (Rendah, Sedang, Tinggi, Sangat Tinggi) |

**Temuan penting:** BAB III Subbab 3.7.4 menyebutkan tujuh indeks atmosfer sebagai fitur pendukung, termasuk **PW**. Kolom `pw` **tidak ditemukan** di dataset ini (fitur yang hilang: {missing_features_line}). Ini adalah gap antara desain metodologi dan dataset aktual yang perlu diklarifikasi sebelum tahap modeling.

**Jumlah missing value per kolom:**

| Kolom | Missing | % |
|---|---|---|
{missing_rows_md}

Seluruh missing value terkonsentrasi pada kolom indeks sounding, konsisten dengan baris `NO_SOUNDING`. Kolom permukaan (`rr`, `tavg`, `rh`) dan kolom label lengkap 100%.

---

## ANALYSIS 2 - Label Distribution

| risk_label | Jumlah | Persentase |
|---|---|---|
{label_dist_rows}
| **Total** | **{fmt_int(n_rows)}** | **100%** |

**Class imbalance ratio (terhadap kelas mayoritas):**

| Kelas | Rasio terhadap {majority_class} |
|---|---|
{imbalance_rows}

- **Kelas mayoritas:** {majority_class} ({fmt_int(int(label_counts.max()))} record, {label_pct[majority_class]:.2f}%)
- **Kelas minoritas:** {minority_class} ({fmt_int(int(label_counts.min()))} record, {label_pct[minority_class]:.2f}%)

Imbalance ratio mayoritas terhadap minoritas mencapai **{imbalance_ratio:.2f} : 1**, tergolong *severe imbalance*.

---

## ANALYSIS 3 - Label Source Audit

| label_source | Jumlah | Persentase |
|---|---|---|
{source_rows}
| **Total** | **{fmt_int(n_rows)}** | **100%** |

**Kontribusi CAPE terhadap seluruh dataset:**
- CAPE secara aktif **mengubah** label pada **{cape_modified_count} record ({cape_contribution_pct:.2f}%)** dari total dataset.
- CAPE **tidak tersedia** pada {cape_missing_count} record boundary zone ({cape_missing_count/n_rows*100:.2f}%), label tetap mengikuti RR.
- Pada {rr_only_count} record ({rr_only_count/n_rows*100:.2f}%) lainnya, label sepenuhnya ditentukan oleh RR saja.

CAPE berperan sebagai modifier minor: kontribusi langsung < 1% dari seluruh label akhir, sesuai desain metodologi BAB III.

---

## ANALYSIS 4 - CAPE Impact Assessment

**Jumlah label berubah karena CAPE:** {n_modified} record ({pct_of_total:.2f}% dari total dataset; {pct_of_boundary:.2f}% dari {boundary_total} record boundary zone).

**Distribusi perubahan per kelas (arah kenaikan):**

| Arah Perubahan | Jumlah |
|---|---|
{direction_rows}

**Kontribusi CAPE terhadap komposisi akhir tiap kelas:**

| Kelas | Total Record | Berasal dari Modifikasi CAPE | % dari Kelas |
|---|---|---|---|
{cape_class_rows}

**Interpretasi kuantitatif:**
- Secara agregat, dampak CAPE terhadap keseluruhan dataset **sangat kecil** ({pct_of_total:.2f}% dari total record).
- Secara relatif per kelas, dampak terbesar terjadi pada kelas **{max_impact_row['risk_label']}** ({max_impact_row['persentase_dari_kelas']:.2f}% dari populasi kelas tersebut berasal dari modifikasi CAPE).
- **Kesimpulan:** CAPE **tidak memiliki dampak signifikan secara statistik** terhadap distribusi label akhir secara keseluruhan, namun memiliki **dampak marjinal yang konsisten dengan desain metodologi** sebagai *conditional risk modifier* pada zona transisi, bukan penentu utama.

---

## ANALYSIS 5 - Modeling Readiness (LSTM Multiclass)

### a. Kesiapan Dataset untuk LSTM Multiclass

| Aspek | Status | Catatan |
|---|---|---|
| Kontinuitas temporal | Siap | {len(missing_dates)} gap tanggal kalender dari {len(full_range)} hari yang diharapkan |
| Kelengkapan label | Siap | 0 label kosong, deterministik sesuai decision rules |
| Kelengkapan fitur permukaan (RR, Tavg, RH) | Siap | 0 missing |
| Kelengkapan fitur sounding (CIN, K-Index, LI, TT, SWEAT) | Perlu perhatian | {int(missing_per_col['cin'])} baris missing ({missing_pct['cin']:.2f}%) |
| Ketersediaan fitur PW sesuai BAB III | Tidak siap | Kolom `pw` tidak ada di dataset |
| Ketersediaan data uji 2025 (sesuai BAB III 3.7.6) | Tidak siap | Dataset hanya sampai {date_max} |

### b. Risiko Akibat Class Imbalance

- Rasio imbalance {imbalance_ratio:.2f}:1 berisiko tinggi menyebabkan model bias memprediksi kelas mayoritas ({majority_class}).
- Tahun **2019 tidak memiliki satu pun** record kelas "Sangat Tinggi" (0 record), sementara tahun 2022 memuat {int(st_per_year.get(2022, 0))} dari {fmt_int(int(label_counts['Sangat Tinggi']))} total kejadian.
- Jarak antar kejadian "Sangat Tinggi": median {gaps.median():.1f} hari, jeda maksimum {gaps.max():.0f} hari (~{gaps.max()/365:.2f} tahun) tanpa satu pun kejadian.

### c. Estimasi Dampak Lookback terhadap Jumlah Sampel

| Lookback Window (W) | Sequence Terpakai | Sampel Hilang | % Hilang dari Total |
|---|---|---|---|
{lookback_rows}

Jika pembagian dilakukan secara kronologis menjadi tiga bagian (train/validation/test), kehilangan sampel berpotensi terjadi di setiap titik potong:

| Lookback Window (W) | Estimasi Total Sampel Hilang (3 titik potong) | % dari Total |
|---|---|---|
{lookback_split_rows}

Dampak lookback terhadap ukuran dataset relatif kecil (< 3,1% untuk window hingga 30 hari), namun perlu diverifikasi ulang pada tahap pembagian dataset karena kehilangan terjadi tepat di awal setiap subset.

### d. Potensi Hilangnya Sampel Minoritas

Simulasi pembagian kronologis 70/15/15 (ilustratif, hanya untuk audit kesiapan - bukan penerapan aktual):

| Subset | Jumlah Baris | Rentang Tanggal | Rendah | Sedang | Tinggi | Sangat Tinggi |
|---|---|---|---|---|---|---|
{split_table_rows}

Kelas "Sangat Tinggi" masih terwakili di ketiga subset, namun jumlahnya sangat kecil secara absolut pada validation dan test, sehingga estimasi metrik evaluasi untuk kelas ini berisiko memiliki varians tinggi.

---

## ANALYSIS 6 - Training Recommendation

**Catatan: rekomendasi berikut bersifat analitis/rencana, tidak diterapkan pada tahap ini.**

### a. Penggunaan `class_weight`
{recommendations['class_weight']}

### b. Penggunaan Focal Loss
{recommendations['focal_loss']}

### c. Perlunya Oversampling
{recommendations['oversampling']}

### d. Strategi Train/Validation/Test Split untuk Time Series
{recommendations['split_strategy']}

---

## Validasi Kepatuhan Batasan Tahap 9

| Batasan | Status |
|---|---|
| Tidak melakukan training model | Dipatuhi |
| Tidak melakukan feature engineering baru | Dipatuhi |
| Tidak melakukan lookback generation (sequence dibuat) | Dipatuhi - hanya estimasi kuantitatif |
| Tidak melakukan balancing/oversampling/undersampling | Dipatuhi |
| Tidak melakukan scaling/normalisasi | Dipatuhi |
| Tidak melakukan penghapusan record | Dipatuhi |
| Data tidak diubah | Dipatuhi - hanya dibaca dan dianalisis |

## Acceptance Criteria

- [x] Tidak ada data yang diubah
- [x] Tidak ada row yang dihapus
- [x] Tidak ada label yang diubah
- [x] Seluruh analisis (1-6) berhasil dibuat

## **STAGE 9 = PASS**
"""

with open(OUTPUT_REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"Laporan disimpan ke: {OUTPUT_REPORT_PATH}")


Laporan disimpan ke: /mnt/user-data/outputs/STAGE9_REPORT.md


In [23]:
# Verifikasi akhir: bukti bahwa data original tidak berubah sama sekali
input_check = pd.read_csv(INPUT_PATH)
unchanged = input_check.equals(df_original)
row_count_unchanged = len(input_check) == n_rows_input

print("=== VERIFIKASI AKHIR TAHAP 9 ===")
print("Dataset input tidak berubah (dibaca ulang == df_original)?:", unchanged)
print("Jumlah baris tetap                                       :", row_count_unchanged, f"({n_rows_input})")
print()
print("STAGE 9 = PASS" if (unchanged and row_count_unchanged) else "STAGE 9 = FAIL")


=== VERIFIKASI AKHIR TAHAP 9 ===
Dataset input tidak berubah (dibaca ulang == df_original)?: True
Jumlah baris tetap                                       : True (2922)

STAGE 9 = PASS
